# El modelo de Kaurix

Entrena el reconocedor de ingredientes y lo deja listo para meter en la app.

Se corre **en Colab con GPU**: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.
En CPU funciona igual pero tarda horas en vez de minutos.

## Qué hace, en orden

1. Lee `modelo/clases.json` del repo: las 55 clases y de dónde sale cada una.
2. Baja fotos de Open Images, una carpeta por clase.
3. Entrena por transferencia sobre MobileNetV3.
4. Exporta un `.tflite` **con metadatos**, que es lo que ML Kit necesita para
   leer los nombres de las clases.

El archivo que sale va a `modules/reconocedor/android/src/main/assets/modelo.tflite`.
El módulo lo detecta solo: no hay que tocar código.

## Por qué Open Images y no fotos de catálogo

Son fotos de gente en escenas reales, no productos sobre fondo blanco. Un
dataset de catálogo entrena un modelo que anda en el catálogo y falla en una
cocina, que es exactamente donde se juega.

Igual queda una brecha: apuntar un teléfono a la mesada no es lo mismo que una
foto de Flickr. La forma de medirla es entrenar con esto y **validar con unas
pocas fotos propias** —ver la última celda—, que dice el número de verdad en
vez de esconderlo.

## 1. Preparativos

In [ ]:
# `fiftyone` baja de Open Images solo lo que se le pide, sin traer los 9 millones
# de imágenes del dataset entero. `tflite-support` es lo que escribe los metadatos
# que ML Kit lee.
!pip install -q fiftyone tflite-support==0.4.4

import json, os, shutil, random, urllib.request
from pathlib import Path

import tensorflow as tf

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "no hay — va a tardar")

In [ ]:
# La especificación de las clases sale del repo, no se copia acá: si se copiara,
# el día que cambie una clase habría dos listas y una quedaría vieja.
CLASES_URL = "https://raw.githubusercontent.com/betianaox/kaurix/main/modelo/clases.json"

with urllib.request.urlopen(CLASES_URL) as r:
    CLASES = json.load(r)["clases"]

print(len(CLASES), "clases")
for c in CLASES[:3]:
    print(" ", c["clase"], "->", [o["nombre"] for o in c["oi"]])

In [ ]:
# Cuántas fotos por clase.
#
# 250 es el punto de partida: menos alcanza para que ande y más no cambia mucho
# a esta escala. Subirlo cuesta tiempo de descarga, no de entrenamiento.
POR_CLASE = 250

# A cuánto se escala cada foto. Es lo que espera MobileNetV3 y lo que la app le
# va a dar: `mirar.ts` saca la foto a 320 puntos y ML Kit la reescala a esto.
LADO = 224

DATOS = Path("/content/datos")
SALIDA = Path("/content/salida")
SALIDA.mkdir(exist_ok=True)

## 2. Bajar las fotos

**Donde hay recuadro, se recorta.** De las 55 clases, 22 tienen caja en Open
Images. Ahí la foto se recorta a la caja, así el objeto llena el cuadro como lo
ve la cámara cuando le apuntás. Las otras 33 van enteras y se apoyan en los
recortes al azar del entrenamiento.

Es una asimetría real y conviene tenerla presente al leer los resultados: esas
22 parten con ventaja.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from PIL import Image


def bajar_clase(c):
    """Deja en `datos/<clase>/` las fotos de una clase. Devuelve cuántas."""
    destino = DATOS / c["clase"]
    if destino.exists() and len(list(destino.glob("*.jpg"))) >= POR_CLASE * 0.8:
        return len(list(destino.glob("*.jpg")))  # ya estaba
    destino.mkdir(parents=True, exist_ok=True)

    nombres = [o["nombre"] for o in c["oi"]]
    guardadas = 0

    for nombre in nombres:
        if guardadas >= POR_CLASE:
            break
        # Primero se intenta con recuadros; si esa clase no los tiene, con la
        # etiqueta de imagen entera.
        for tipo in ("detections", "classifications"):
            if guardadas >= POR_CLASE:
                break
            try:
                ds = foz.load_zoo_dataset(
                    "open-images-v7",
                    split="train",
                    label_types=[tipo],
                    classes=[nombre],
                    max_samples=POR_CLASE - guardadas,
                    only_matching=True,
                    shuffle=True,
                    dataset_name=f"tmp_{c['clase']}_{tipo}_{guardadas}",
                )
            except Exception as e:
                print("   ", nombre, tipo, "->", type(e).__name__)
                continue

            for s in ds:
                if guardadas >= POR_CLASE:
                    break
                try:
                    img = Image.open(s.filepath).convert("RGB")
                except Exception:
                    continue

                recortes = []
                if tipo == "detections" and s.ground_truth is not None:
                    for d in s.ground_truth.detections:
                        if d.label != nombre:
                            continue
                        x, y, w, h = d.bounding_box  # relativos, 0..1
                        # Cajas muy chicas dan recortes borrosos: no sirven.
                        if w < 0.10 or h < 0.10:
                            continue
                        W, H = img.size
                        # Un poco de aire alrededor: un recorte al ras del objeto
                        # no se parece a lo que se ve por la cámara.
                        aire = 0.12
                        x0 = max(0, int((x - w * aire) * W))
                        y0 = max(0, int((y - h * aire) * H))
                        x1 = min(W, int((x + w * (1 + aire)) * W))
                        y1 = min(H, int((y + h * (1 + aire)) * H))
                        if x1 - x0 > 40 and y1 - y0 > 40:
                            recortes.append(img.crop((x0, y0, x1, y1)))

                for r in (recortes or [img]):
                    if guardadas >= POR_CLASE:
                        break
                    r.convert("RGB").resize((LADO, LADO)).save(
                        destino / f"{guardadas:04d}.jpg", quality=90
                    )
                    guardadas += 1

            ds.delete()
            if tipo == "detections" and guardadas > 0:
                break  # tenía recuadros, no hace falta la otra vía

    return guardadas


cuenta = {}
for i, c in enumerate(CLASES, 1):
    n = bajar_clase(c)
    cuenta[c["clase"]] = n
    print(f"{i:2}/{len(CLASES)}  {c['clase']:<16} {n}")

## 3. Ver qué quedó

Antes de entrenar hay que mirar el reparto. Una clase con veinte fotos contra
otra con doscientas cincuenta no se aprende: se aprende la de doscientas
cincuenta y la otra queda de adorno.

In [ ]:
flojas = {k: v for k, v in sorted(cuenta.items(), key=lambda x: x[1]) if v < POR_CLASE * 0.5}

print("total de fotos:", sum(cuenta.values()))
print("promedio por clase:", round(sum(cuenta.values()) / len(cuenta)))
print()
if flojas:
    print("MENOS DE LA MITAD DE LO PEDIDO:")
    for k, v in flojas.items():
        print(f"   {k:<16} {v}")
    print()
    print("Esas clases van a andar peor. Opciones: buscarles otra etiqueta de")
    print("Open Images en clases.json, o sacarlas y resolverlas por escena y color.")
else:
    print("todas las clases pasaron la mitad de lo pedido")

## 4. Entrenar

In [ ]:
# Un quinto para validar. Sale del mismo pozo, así que **no mide** cómo anda con
# fotos de una cocina de verdad: mide que aprendió lo que se le mostró. Lo otro
# se mide en la última celda.
entrena = tf.keras.utils.image_dataset_from_directory(
    DATOS, validation_split=0.2, subset="training", seed=1234,
    image_size=(LADO, LADO), batch_size=32, label_mode="categorical",
)
valida = tf.keras.utils.image_dataset_from_directory(
    DATOS, validation_split=0.2, subset="validation", seed=1234,
    image_size=(LADO, LADO), batch_size=32, label_mode="categorical",
)

# El orden de las clases lo fija Keras al leer las carpetas, y **ese mismo orden**
# tiene que ir en los metadatos: si se desfasan, el modelo acierta y la app
# muestra otra cosa.
NOMBRES = entrena.class_names
print(len(NOMBRES), "clases:", NOMBRES)

In [ ]:
# Lo que ensancha el dataset sin más fotos.
#
# El recorte al azar es el que más importa acá: es lo que suple los recuadros que
# 33 de las 55 clases no tienen. El brillo y el contraste imitan la diferencia
# entre luz de tarde y luz de tubo, que es donde se va a jugar.
aumentos = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.25),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.25, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.25),
], name="aumentos")

AUTO = tf.data.AUTOTUNE
entrena = entrena.map(lambda x, y: (aumentos(x, training=True), y), num_parallel_calls=AUTO).prefetch(AUTO)
valida = valida.cache().prefetch(AUTO)

In [ ]:
# MobileNetV3Small: el más chico que da una precisión razonable. El modelo va
# adentro del APK y corre en teléfonos de gama media, así que el tamaño no es un
# detalle.
#
# `include_preprocessing=True` deja la normalización **adentro del modelo**. Es lo
# que permite que los metadatos digan media 0 y desvío 1: la app entrega píxeles
# de 0 a 255 y el modelo se arregla solo. Normalizar de los dos lados es el error
# clásico y da un modelo que acierta acá y falla en el teléfono.
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(LADO, LADO, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=True,
)
base.trainable = False

modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LADO, LADO, 3)),
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(NOMBRES), activation="softmax"),
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
modelo.summary()

In [ ]:
# Primera vuelta: solo la cabeza. La base viene sabiendo mirar y lo que falta es
# que aprenda a nombrar estas 55 cosas.
modelo.fit(entrena, validation_data=valida, epochs=12)

In [ ]:
# Segunda vuelta: se descongela la parte de arriba de la base y se sigue con paso
# corto. Es de donde sale la diferencia entre "distingue fruta de piedra" y
# "distingue una pera de una manzana".
#
# Paso corto de verdad: con el de antes, la base olvida en dos lotes lo que
# aprendió en un millón de fotos.
base.trainable = True
for capa in base.layers[:-40]:
    capa.trainable = False

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
historia = modelo.fit(entrena, validation_data=valida, epochs=10)

## 5. Qué clases confunde

El promedio no sirve para decidir: un 80% puede ser ochenta clases perfectas y
diez irrecuperables. Lo que hay que ver es **cuáles** fallan, porque de ahí sale
si conviene sacarlas y resolverlas por escena y color, como ya se hace con las
gemas y la despensa.

In [ ]:
import numpy as np

ciertas, predichas = [], []
for x, y in valida:
    p = modelo.predict(x, verbose=0)
    ciertas.extend(np.argmax(y.numpy(), axis=1))
    predichas.extend(np.argmax(p, axis=1))

ciertas, predichas = np.array(ciertas), np.array(predichas)

print("acierto general:", round(float((ciertas == predichas).mean()) * 100, 1), "%")
print()
print("las diez peores:")
filas = []
for i, n in enumerate(NOMBRES):
    suyas = ciertas == i
    if not suyas.any():
        continue
    acierto = float((predichas[suyas] == i).mean())
    # Con qué se confunde más.
    otras = predichas[suyas & (predichas != i)]
    conquien = NOMBRES[np.bincount(otras).argmax()] if len(otras) else "—"
    filas.append((acierto, n, conquien, int(suyas.sum())))

for acierto, n, conquien, cuantas in sorted(filas)[:10]:
    print(f"   {n:<16} {acierto*100:5.1f}%  ({cuantas} fotos)  se confunde con {conquien}")

## 6. Exportar para ML Kit

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = conv.convert()

crudo = SALIDA / "modelo_sin_metadatos.tflite"
crudo.write_bytes(tflite)
print(round(len(tflite) / 1e6, 2), "MB")

In [ ]:
# Los metadatos son lo que hace que ML Kit devuelva "manzana" y no el número 0.
# Sin esto el modelo carga igual y `label.text` viene vacío.
#
# media 0 y desvío 1 porque la normalización ya está adentro del modelo. Ver la
# celda donde se arma.
from tflite_support.metadata_writers import image_classifier, writer_utils

etiquetas = SALIDA / "etiquetas.txt"
etiquetas.write_text("\n".join(NOMBRES))

escritor = image_classifier.MetadataWriter.create_for_inference(
    writer_utils.load_file(str(crudo)),
    input_norm_mean=[0.0],
    input_norm_std=[1.0],
    label_file_paths=[str(etiquetas)],
)

final = SALIDA / "modelo.tflite"
writer_utils.save_file(escritor.populate(), str(final))
print(escritor.get_metadata_json())

In [ ]:
# Que el archivo que se va a copiar de verdad funcione, antes de copiarlo.
interp = tf.lite.Interpreter(model_path=str(final))
interp.allocate_tensors()
ent = interp.get_input_details()[0]
sal = interp.get_output_details()[0]
print("entrada:", ent["shape"], ent["dtype"].__name__)
print("salida :", sal["shape"], sal["dtype"].__name__)
assert sal["shape"][-1] == len(NOMBRES), "la salida no tiene una casilla por clase"

# Una foto del propio dataset, para ver que predice algo con sentido.
import glob
una = random.choice(glob.glob(str(DATOS / "*" / "*.jpg")))
esperada = Path(una).parent.name
img = tf.keras.utils.img_to_array(tf.keras.utils.load_img(una, target_size=(LADO, LADO)))
interp.set_tensor(ent["index"], np.expand_dims(img, 0).astype(ent["dtype"]))
interp.invoke()
p = interp.get_tensor(sal["index"])[0]
print()
print("foto de:", esperada, "-> dice:", NOMBRES[int(np.argmax(p))], f"({p.max():.0%})")

In [ ]:
from google.colab import files
files.download(str(final))
print("Va en modules/reconocedor/android/src/main/assets/modelo.tflite")
print("El módulo lo detecta solo: no hay que tocar código.")

## 7. Lo que este cuaderno NO mide

El acierto de arriba sale de fotos de Open Images, o sea del mismo pozo del que
salió el entrenamiento. Dice que aprendió lo que se le mostró, **no** que ande
apuntando un teléfono a una mesada.

Para saber eso hace falta un puñado de fotos propias —veinte por clase de las
que más importan alcanza— sacadas con el teléfono, en una cocina, con la luz de
siempre. Se dejan en `/content/propias/<clase>/` y se corre la celda de abajo.

Si el número cae mucho respecto del de arriba, esa es la brecha entre la foto
descargada y la foto real, y ahí se decide: más aumentos, más fotos propias, o
sacar las clases que no se sostienen.

In [ ]:
PROPIAS = Path("/content/propias")

if not PROPIAS.exists() or not any(PROPIAS.iterdir()):
    print("Todavía no hay fotos propias. Subir a /content/propias/<clase>/ y volver.")
else:
    prueba = tf.keras.utils.image_dataset_from_directory(
        PROPIAS, image_size=(LADO, LADO), batch_size=32, label_mode="categorical",
        class_names=NOMBRES,  # el mismo orden, o los números no significan lo mismo
    )
    perdida, acierto = modelo.evaluate(prueba, verbose=0)
    print("con fotos propias:", round(acierto * 100, 1), "%")